# Stage B — All Experiments (Sci-Sentence + FWS + SAL)

Reproduces the actual Stage-B experiments run on the GPU: the **frozen encoder sweep**, the **full
fine-tune comparison**, and the **balancing sweep** — all evaluated end-to-end on the 10 gold arXiv
papers (49 gaps).

**Stage B** = (B.1) cue rules + (B.2) a classifier. We keep a sentence if a rule fires **or** the
classifier predicts limitation/future_work. The *frozen* head = a sentence encoder (not trained) +
a class-weighted logistic regression (trained).

**Data** (cached in `data/external/`): Sci-Sentence (gap/limitation/future_work + neg), FWS
(future_work + neg), SAL/RCT (limitation + neg). Label map: Sci LIMITATION→limitation,
EXTENSION & RESEARCH GAP→future_work; FWS 1→future_work; SAL True→limitation; else none.

**Compute:** uses GPU if available, else CPU (slower; embedding is cached per encoder in-session).
Reference numbers from the H100 runs are in `v100_results/`.

## Setup: helpers + loaders + eval

In [ ]:
# ===== Stage B helpers (self-contained; stdlib + sklearn + sentence-transformers) =====
import re, json
import numpy as np, pandas as pd
from pathlib import Path

# locate repo root + data
ROOT = Path.cwd()
while not (ROOT/"data"/"external").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT/"data"
EXT  = DATA/"external"
print("data dir:", EXT)

# ---- text / matching / cue rules (Stage B.1) ----
_W=re.compile(r"[a-z0-9]+"); _DEH=re.compile(r"(\w)-\s*(\w)")
_PH=re.compile(r"\{\{[^}]*\}\}"); _MA=re.compile(r"\\\(.*?\\\)|\\\[.*?\\\]|\$[^$]*\$"); _NW=re.compile(r"[^a-z0-9'\- ]+"); _WS=re.compile(r"\s+")
def norm(s): s=(s or "").lower();s=_PH.sub(" ",s);s=_MA.sub(" ",s);s=_NW.sub(" ",s);return _WS.sub(" ",s).strip()
def token_containment(n,h):
    nt=_W.findall(norm(_DEH.sub(r"\1\2",n)))
    if not nt: return 0.0
    hs=set(_W.findall(norm(_DEH.sub(r"\1\2",h)))); return sum(1 for t in nt if t in hs)/len(nt)
_FUT=re.compile(r"\b(future work|future direction|future research|future stud(?:y|ies)|in future|for future|as future|future investigation|leave[sd]?\s+(?:\w+\s+){0,6}?(?:for|to|as)\s+future|left\s+(?:\w+\s+){0,4}?future|remains?\s+(?:\w+\s+){0,4}?(?:future|open|interesting|valuable)|we\s+(?:plan|intend|aim|hope|would like|will)\s+to|we\s+(?:will|plan to|intend to)\s+explore|next step|further (?:work|research|investigation|study)|would be (?:interesting|valuable|worthwhile)|it would be interesting|an? (?:interesting|promising|natural) (?:direction|next step|avenue)|(?:plan|plans|planning)\s+to|intend(?:s|ed)?\s+to|also\s+intend|future directions? include|future plans|plans? include|we\s+will\s+(?:investigate|explore|extend|incorporate|study|consider|try|continue|apply|also)|will\s+(?:investigate|extend|incorporate|study|continue|consider)|to extend (?:our|the) (?:work|approach|method|model|framework|analysis|study)|another (?:interesting|promising|natural) (?:direction|avenue|line)|also (?:want|hope|interested|would like|plan|like) to|(?:promising|interesting|natural|fruitful) (?:future )?direction is|direction is to|investigate whether)\b",re.I)
_LIM=re.compile(r"\b(limitation|shortcoming|drawback|weakness|caveat|we did not|we do not|we have not|did not (?:study|consider|explore|address|evaluate|test)|is (?:restricted|limited|confined) to|are (?:restricted|limited|confined) to|only (?:considers?|applies?|works?|handles?)|remains? (?:unexplored|unaddressed|an open|unclear|unknown|challenging|elusive)|unexplored|unaddressed|beyond the scope|out of scope|assumes? (?:the (?:availability|existence)|access to)|(?:cannot|can not|fail[s]? to|do(?:es)? not (?:generalize|scale|extend))|not (?:sufficiently|fully|yet) (?:explored|studied|understood|addressed|clear)|lack[s]? (?:of|a )|adds? (?:latency|cost|overhead|complexity)|(?:suffers? from|prone to|sensitive to|struggles? with)|(?:formidable|profound|significant) (?:challenge|obstacle|difficulty))\b",re.I)
_CG=re.compile(r"\b(we (?:show|prove|establish|demonstrate|present|propose|introduce|derive)|in this (?:paper|work|section) we|our (?:main )?(?:result|theorem|contribution))\b",re.I)
def cue_label(s):
    f=_FUT.search(s); l=_LIM.search(s)
    if not f and not l: return None
    if f and not l: return "future_work"
    if l and not f:
        if _CG.search(s) and "limitation of our" not in s.lower():
            return "limitation" if re.search(r"\bour\b|\bthis (?:work|paper|approach|method|study)\b",s,re.I) else None
        return "limitation"
    return "future_work"
LABELS=["none","limitation","future_work"]

In [ ]:
# ---- dataset loaders -> rows {sentence,label,ds,group} ----
def load_sci(researchgap_as="future_work"):
    df=pd.read_excel(EXT/"scisentence_repo/datasets/Based_Version.xlsx")
    M={"LIMITATION":"limitation","EXTENSION":"future_work","RESEARCH GAP":researchgap_as,
       "OVERALL":"none","DESCRIPTION":"none","RESULT":"none","OTHER":"none"}
    return [{"sentence":str(r.Sentence),"label":M.get(str(r.Category).upper(),"none"),
             "ds":"sci","group":"sci::"+str(getattr(r,'Title',r.Id))[:40]} for r in df.itertuples()]
def load_fws(pos=2000,neg=4000):
    df=pd.read_csv(EXT/"fws_corpus.csv"); df["lab"]=df["label"].map({1:"future_work",0:"none"})
    p=df[df.lab=="future_work"].sample(min(pos,(df.lab=="future_work").sum()),random_state=0)
    n=df[df.lab=="none"].sample(min(neg,(df.lab=="none").sum()),random_state=0)
    return [{"sentence":str(r.text),"label":r.lab,"ds":"fws","group":"fws::"+str(r.id)} for r in pd.concat([p,n]).itertuples()]
def load_sal(neg=2000):
    df=pd.read_csv(EXT/"sal_rct/train.csv")
    df["lab"]=df["limitation_annotation"].astype(str).str.contains("True").map({True:"limitation",False:"none"})
    p=df[df.lab=="limitation"]; n=df[df.lab=="none"].sample(min(neg,(df.lab=="none").sum()),random_state=0)
    return [{"sentence":str(r.sentences),"label":r.lab,"ds":"sal","group":"sal::"+str(r.pmid)} for r in pd.concat([p,n]).itertuples()]

# gold eval (precomputed Stage-A candidates + gold gaps)
gold_eval=json.load(open(EXT/"gold_eval.json",encoding="utf-8"))
GS=[g["gap_sentence"] for g in gold_eval["gold"]]
def build_df(researchgap_as="future_work"):
    df=pd.DataFrame(load_sci(researchgap_as)+load_fws()+load_sal())
    df=df[df.sentence.str.len()>=20].reset_index(drop=True)
    df=df[~df.sentence.map(lambda s:max((token_containment(g,s) for g in GS),default=0)>=0.8)].reset_index(drop=True)
    return df

from sklearn.model_selection import GroupShuffleSplit
def split_per_dataset(df):
    tr=[]; val={}
    for ds in ["sci","fws","sal"]:
        sub=df[df.ds==ds].reset_index(drop=True)
        tri,vai=next(GroupShuffleSplit(1,test_size=0.15,random_state=0).split(sub,sub.label,sub.group))
        tr.append(sub.iloc[tri]); val[ds]=sub.iloc[vai].reset_index(drop=True)
    return pd.concat(tr).reset_index(drop=True), val

In [ ]:
# ---- frozen head + gold eval ----
import torch
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
print("device:",DEVICE)
_enc_cache={}
def get_encoder(name):
    if name not in _enc_cache: _enc_cache[name]=SentenceTransformer(name,device=DEVICE)
    return _enc_cache[name]
def embed(name,sents):
    if not len(sents): return np.zeros((0,384))
    return np.asarray(get_encoder(name).encode(list(sents),normalize_embeddings=True,batch_size=256,show_progress_bar=False))

cand=gold_eval["candidates"]; gold=pd.DataFrame(gold_eval["gold"])
def gold_recall(clf, encoder_name, use_rules=True):
    allc=[s for pid in cand for s in cand[pid]]; E=embed(encoder_name,allc)
    k=0; cemb={}
    for pid in cand: cemb[pid]=E[k:k+len(cand[pid])]; k+=len(cand[pid])
    kept=0; rec={"all":[0,0],"limitation":[0,0],"future_work":[0,0]}
    for pid in cand:
        hl=[list(clf.classes_)[int(r.argmax())] for r in clf.predict_proba(cemb[pid])] if len(cand[pid]) else []
        keep=[s for i,s in enumerate(cand[pid]) if (use_rules and cue_label(s)) or hl[i] in ("limitation","future_work")]
        kept+=len(keep); kt=" ".join(keep)
        for g in gold[gold.paper_id==pid].itertuples():
            hit=token_containment(g.gap_sentence,kt)>=0.7; rec["all"][1]+=1; rec["all"][0]+=int(hit)
            if g.gap_type in rec: rec[g.gap_type][1]+=1; rec[g.gap_type][0]+=int(hit)
    return {"kept":kept,**{f"recall_{k}":round(h/n,3) if n else None for k,(h,n) in rec.items()}}
def macro(y,p):
    present=[l for l in LABELS if l in set(y)]
    return round(f1_score(y,p,labels=present,average="macro",zero_division=0),3)

def train_frozen(encoder_name, tr, val, class_weight="balanced"):
    clf=LogisticRegression(C=2.0,max_iter=2000,class_weight=class_weight).fit(embed(encoder_name,tr.sentence.tolist()),tr.label.values)
    vm={ds:macro(val[ds].label.values,clf.predict(embed(encoder_name,val[ds].sentence.tolist()))) for ds in val}
    return clf, vm

## Build data + per-dataset paper-disjoint split

In [ ]:
df=build_df()
print("rows per dataset:",df.ds.value_counts().to_dict())
print("label x dataset:",df.groupby(['ds','label']).size().to_dict())
tr,val=split_per_dataset(df)
print("train",len(tr),"| val per ds:",{k:len(v) for k,v in val.items()})

## Experiment 1 — frozen encoder sweep
Same data + logreg, only the encoder changes. (Default: two encoders for a quick run; add the others
to reproduce the full sweep — mpnet/bge-base/e5 downloaded on first use.)

In [ ]:
ENCODERS = {
    "bge-small":"BAAI/bge-small-en-v1.5",
    "MiniLM":"sentence-transformers/all-MiniLM-L6-v2",
    # "bge-base":"BAAI/bge-base-en-v1.5",
    # "mpnet":"sentence-transformers/all-mpnet-base-v2",
    # "e5-base":"intfloat/e5-base-v2",
}
rows=[]
for name,hf in ENCODERS.items():
    clf,vm=train_frozen(hf,tr,val)
    g=gold_recall(clf,hf,use_rules=True)
    rows.append({"encoder":name,"val_overall":macro(pd.concat(val.values()).label.values,
                 clf.predict(embed(hf,pd.concat(val.values()).sentence.tolist()))),
                 **vm,"gold_recall_all":g["recall_all"],"gold_lim":g["recall_limitation"],
                 "gold_fw":g["recall_future_work"],"kept":g["kept"]})
sweep=pd.DataFrame(rows).set_index("encoder"); sweep

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(7,4))
sweep[["gold_recall_all","gold_lim","gold_fw"]].plot(kind="bar",ax=ax)
ax.set_ylabel("gold recall (rules+head)"); ax.set_ylim(0,1); ax.set_title("Encoder sweep — gold recall")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

### Reference (H100, full 5-encoder sweep)
| encoder | gold recall_all | limitation | future_work | kept |
|---|---|---|---|---|
| **MiniLM** | **0.571** | 0.636 | 0.857 | 170 |
| mpnet | 0.571 | 0.455 | 0.786 | 186 |
| bge-small | 0.429 | 0.545 | 0.786 | 97 |
| bge-base / e5-base | 0.408 | 0.455 | 0.786 | 71–76 |

**MiniLM-L6 wins** (best recall + best limitation/future_work). Bigger ≠ better.

## Experiment 2 — full fine-tune comparison (reference)
Full fine-tuning (all weights) of DistilBERT / SciBERT / PubMedBERT on the same data. These **overfit**:
they top in-distribution validation but lose on gold. Code is included (set `RUN_FT=True`, needs GPU +
torch≥2.6 for SciBERT/PubMedBERT safetensors). Default off; reference numbers shown below.

In [ ]:
RUN_FT = False  # set True on a GPU box (torch>=2.6) to actually fine-tune
# Reference results from the H100 runs (gold rules+head / in-dataset overall macro-F1):
ft_ref = pd.DataFrame([
 {"model":"frozen MiniLM+SAL","val_macroF1":0.656,"gold_recall_all":0.571,"gold_lim":0.636},
 {"model":"DistilBERT FT","val_macroF1":0.745,"gold_recall_all":0.306,"gold_lim":0.182},
 {"model":"SciBERT FT","val_macroF1":0.846,"gold_recall_all":0.327,"gold_lim":0.273},
 {"model":"PubMedBERT FT","val_macroF1":0.890,"gold_recall_all":0.327,"gold_lim":0.364},
]).set_index("model")
print("Fine-tunes: high validation, low gold -> overfitting. Frozen wins on gold.")
ft_ref

## Experiment 3 — balancing sweep (MiniLM frozen)
Does data-level balancing help beyond the class-weighting we already use?

In [ ]:
MINILM="sentence-transformers/all-MiniLM-L6-v2"
val_all=pd.concat(val.values()).reset_index(drop=True)
def quick(cw,trn):
    clf=LogisticRegression(C=2.0,max_iter=2000,class_weight=cw).fit(embed(MINILM,trn.sentence.tolist()),trn.label.values)
    return gold_recall(clf,MINILM,True)
# A balanced | B natural | C resample 2:1.5:1
A=quick("balanced",tr); B=quick(None,tr)
nlim=(tr.label=="limitation").sum(); rng=np.random.RandomState(0); idx=[]
for lab,m in [("limitation",1.0),("future_work",1.5),("none",2.0)]:
    pool=tr[tr.label==lab]; idx.append(pool.sample(min(int(nlim*m),len(pool)),random_state=0))
C=quick(None,pd.concat(idx))
print("A class_weight=balanced:",A)
print("B natural (no weight)  :",B)
print("C resample 2:1.5:1     :",C)

### Balancing finding
A (class-weighted) **0.571** · B (no weight) 0.265 · C (resample) 0.551 · D (+aug paraphrases, H100) 0.571.
→ **class-weighting already balances; extra resampling/augmentation gives no gold gain.** The real lever
is *new* limitation data, not reshuffling.

## Experiment 4 — Stage B.1 cue-rule optimization (mined on train, validated held-out)

The cue rules (`_FUT` / `_LIM`) are high-precision regexes. We optimized them **per class on each
dataset's train split**, then measured on the **held-out test split** (never touched during mining —
avoiding the FWS-leakage trap). The inline `_FUT` above already includes the shipped additions.

**future_work → FWS** (paper-disjoint 80/10/10):

| cue rules | P | R | F1 |
|---|---|---|---|
| before | 0.843 | 0.448 | 0.585 |
| **after (shipped)** | 0.819 | 0.538 | **0.650** |

Added 10 generalizable intent groups (`plan(ning) to`, `intend(s/ed) to`, `future directions include`,
`we will investigate/extend/…`, `to extend our/the work`, `another promising direction`,
`also want/hope to`, `investigate whether`, …). Recall **+0.090** at only **−0.024** precision.

We deliberately **excluded** bare `future` / `the future` — they score ~0.98 on FWS *only because the
whole corpus is future-work sentences*, and over-fire on general text. The greedy-but-leaky variant that
kept them reached 0.694; the curated set (0.650) is the honest, transferable gain.

**limitation → SAL**: cue mining yields almost nothing (F1 0.267 → 0.275). Biomedical limitations are
**diffuse and cue-less** ("longer follow-up is needed", "not blinded", "evaluated by investigators"), so
no high-precision lexical trigger exists — the embedding head / Stage C handle limitation, not the cue layer.

## Summary
- **Best Stage-B head: frozen MiniLM-L6 + logreg** on Sci-Sentence+FWS+SAL → gold recall **0.571**
  (limitation 0.636, future_work 0.857).
- **Full fine-tunes overfit** (validation up, gold down). **Balancing is already handled** by class weights.
- SAL fixed the limitation bottleneck; FWS fixed future_work. Full logs in `v100_results/`.

- **Cue rules were tuned on train, validated held-out (Exp 4):** future_work F1 0.585→0.650 on FWS; limitation stays cue-less on SAL (head/Stage C territory).